# MOFSemNet — server experiment notebook (single file)

Self-contained: it does **not** import anything from the MOFSemNet repository. Everything
(data loading, features, graph building, Black Hole / random / k-center sparsification, the GNNs,
the baselines, result export) is in this notebook, so the whole thing can be deleted later.

**What to upload next to this notebook**
1. `MOFCSD.csv` (from `data/raw/` in the repo)
2. `chemberta_embeddings.npz` (from `data/embeddings/`) — optional but recommended. If missing,
   the notebook computes the embeddings itself (needs internet access to Hugging Face).
3. `molt5_embeddings.npz` — only if you want MolT5.

**How to run:** edit the *Configuration* cell (pick a `PROFILE`), then *Run All*.
It prints how many runs it will do first. Results are appended to a CSV after **every single run**,
so if the server kills the job you just run the notebook again and it **resumes** where it stopped.

**Outputs** (folder `mofsemnet_results/`, also zipped for download):
`per_run_results.csv` (one row per run + seed), `aggregated_results.csv` (mean ± std over seeds),
`graph_vs_no_graph.csv`, `mofsemnet_results.xlsx` (all sheets), `run_metadata.json`, `README.txt`.

**Reproducibility:** every row stores the seed, the test-set name, the exact configuration and the
fingerprint (SHA-256) of the data file. Same notebook + same data + same seeds = same numbers
(GPU runs can differ in the last decimals because some GPU operations are not bit-exact).
**Extra experiments (PROFILE `extra_smoke` / `extra`).** Adds, as new switches that are OFF in every older profile
(old run ids and numbers are unchanged): XGBoost / LightGBM baselines, a log1p target, a no-LCD/LFS ablation,
metal and RDKit linker descriptors, linker-level de-duplicated and descriptor-space graphs, and graph-smoothed
features for XGBoost. Results go to `server_extra_<date>/`. See section 7b.


## 1. Install / imports

In [2]:
# Uncomment on a fresh server (PyTorch Geometric needs a torch build that matches your CUDA):
!pip install -q pandas numpy networkx scikit-learn rdkit tqdm openpyxl transformers xgboost lightgbm pymatgen
!pip install -q torch_geometric

import hashlib, itertools, json, os, platform, random, sys, time, warnings, zipfile
from datetime import datetime

import networkx as nx
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from networkx.algorithms.community import louvain_communities
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, Descriptors, rdFingerprintGenerator, rdMolDescriptors
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from torch_geometric.nn import GATConv, GCNConv
from tqdm.auto import tqdm

try:
    import xgboost as xgb
except ImportError:
    xgb = None
try:
    import lightgbm as lgb
except ImportError:
    lgb = None

RDLogger.DisableLog("rdApp.*")
warnings.filterwarnings("ignore")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE, "| torch", torch.__version__)


[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python3 -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python3 -m pip install --upgrade pip
device: cuda | torch 2.9.1+cu128


## 2. Configuration  ← the only cell you normally need to edit

Pick a `PROFILE`, then optionally override single settings in the `CFG[...] = ...` lines below it.

| profile | meaning | rough size |
|---|---|---|
| `smoke` | 800 MOFs, 2 seeds, tiny grid — checks the notebook works | a few minutes |
| `quick` | 3 seeds, k ∈ {3,10}, 3 sparsification methods | hours on CPU, much less on GPU |
| `standard` | 5 seeds, k ∈ {3,5,10}, all methods, retention 30/50 % | the main run |
| `full` | 10 seeds, k ∈ {3,5,10}, retention 30/50/70 %, 3 Black Hole weightings, hybrid β ∈ {0.25,0.5,0.75}, scaling on/off, 2 test sets, GAT | very large — trim it first |

In [3]:
PROFILE = "extra"                          # "smoke" | "quick" | "standard" | "full" | "extra_smoke" | "extra"

DATA_PATH = "MOFCSD.csv"
EMB_PATHS = {"chemberta": "chemberta_embeddings.npz", "molt5": "molt5_embeddings.npz"}
HF_IDS = {"chemberta": "seyonec/ChemBERTa-zinc-base-v1", "molt5": "laituan245/molt5-base"}
EXTRA_RUN_DATE = "2026-11-20"             # part of the extra output folder name; keep fixed so a re-run resumes
OUT_DIR = f"server_extra_{EXTRA_RUN_DATE}" if PROFILE.startswith("extra") else "mofsemnet_results_molt5"

PROFILES = {
    "smoke": dict(
        n_subsample=800, seeds=[0, 1], language_models=["chemberta"], test_sets=["random15"], scalings=["none"],
        feature_sets=["fingerprint", "fingerprint+embedding"], node_features=["fingerprint"],
        non_graph_models=["random_forest", "knn"], gnn_models=["gcn", "residual_gcn"],
        graph_types=["cs2", "semantic"], k_values=[3], hybrid_betas=[0.5],
        sparsify_graphs=["cs2", "semantic"], sparsify_k_values=[3],
        sparsify_methods=["black_hole", "random", "k_center"], retentions=[0.5],
        bh_weights=[(0.33, 0.33, 0.33)], epochs=30),
    "quick": dict(
        n_subsample=None, seeds=[0, 1, 2], language_models=["chemberta"], test_sets=["random15"], scalings=["none"],
        feature_sets=["fingerprint", "embedding", "fingerprint+embedding"], node_features=["fingerprint"],
        non_graph_models=["random_forest", "mlp"], gnn_models=["gcn", "graphsage", "residual_gcn"],
        graph_types=["cs2", "fingerprint", "semantic", "hybrid"], k_values=[3, 10], hybrid_betas=[0.5],
        sparsify_graphs=["cs2", "fingerprint", "semantic", "hybrid"], sparsify_k_values=[3],
        sparsify_methods=["black_hole", "random", "k_center"], retentions=[0.3, 0.5],
        bh_weights=[(0.33, 0.33, 0.33)], epochs=200),
    "standard": dict(
        n_subsample=None, seeds=[0, 1, 2, 3, 4], language_models=["chemberta"], test_sets=["random15"], scalings=["none"],
        feature_sets=["fingerprint", "embedding", "fingerprint+embedding"], node_features=["fingerprint"],
        non_graph_models=["random_forest", "mlp", "knn"], gnn_models=["gcn", "graphsage", "residual_gcn"],
        graph_types=["cs2", "fingerprint", "semantic", "hybrid"], k_values=[3, 5, 10], hybrid_betas=[0.5],
        sparsify_graphs=["cs2", "fingerprint", "semantic", "hybrid"], sparsify_k_values=[3, 10],
        sparsify_methods=["black_hole", "random", "k_center"], retentions=[0.3, 0.5],
        bh_weights=[(0.33, 0.33, 0.33)], epochs=200),
    "full": dict(
        n_subsample=None, seeds=list(range(10)), language_models=["chemberta", "molt5"],
        test_sets=["random15", "cs2_hubs"], scalings=["none", "standard"],
        feature_sets=["fingerprint", "embedding", "fingerprint+embedding"],
        node_features=["fingerprint", "fingerprint+embedding"],
        non_graph_models=["random_forest", "mlp", "knn"], gnn_models=["gcn", "graphsage", "gat", "residual_gcn"],
        graph_types=["cs2", "fingerprint", "semantic", "hybrid"], k_values=[3, 5, 10], hybrid_betas=[0.25, 0.5, 0.75],
        sparsify_graphs=["cs2", "fingerprint", "semantic", "hybrid"], sparsify_k_values=[3, 5, 10],
        sparsify_methods=["black_hole", "random", "k_center"], retentions=[0.3, 0.5, 0.7],
        bh_weights=[(0.33, 0.33, 0.33), (0.6, 0.2, 0.2), (0.2, 0.6, 0.2)], epochs=200),
}
_EXTRA_COMMON = dict(          # the old blocks are off; only the extra block runs
    language_models=[], test_sets=["random15", "cs2_hubs"], scalings=["none"], feature_sets=[], node_features=[],
    non_graph_models=[], gnn_models=[], graph_types=[], k_values=[], hybrid_betas=[0.5], sparsify_graphs=[],
    sparsify_k_values=[], sparsify_methods=[], retentions=[], bh_weights=[(0.33, 0.33, 0.33)])
PROFILES["extra_smoke"] = dict(_EXTRA_COMMON, n_subsample=800, seeds=[0, 1], epochs=30)
PROFILES["extra"] = dict(_EXTRA_COMMON, n_subsample=None, seeds=[0, 1, 2, 3, 4], epochs=200)
CFG = dict(PROFILES[PROFILE])

# ---- Settings shared by every profile (edit freely) -------------------------------------------
CFG.update(
    # which experiment blocks to run
    block_non_graph=True,     # RF / MLP / kNN on fingerprint, embedding, fingerprint+embedding  (no graph)
    block_no_edge=True,       # the GNNs with NO edges  -> the "does the graph help or hurt?" reference
    block_graph_full=True,    # GNNs on the full (unpruned) graphs, every graph type x k
    block_sparsify=True,      # Black Hole / random / k-center pruning on the chosen graphs
    # GNN training settings (same as the repo's Step 5)
    hidden=64, lr=0.005, weight_decay=1e-3, patience=20,
    # baselines
    rf_trees=100, knn_neighbors=5, mlp_hidden=(128, 64),
    # splits: 15 % test (fixed, split_state=42), 15 % of the rest for early stopping
    split_state=42, test_frac=0.15, val_frac=0.15,
    # safety valve: stop politely after this many hours (re-run the notebook to resume). None = no limit
    max_hours=None,
    # ---- extra experiments (default OFF; switched on by the "extra" profiles below) ----
    block_extra=False,
    extra_groups=["g1_baselines", "g2_log", "g3_nopore", "g4_descriptors", "g5_graphs", "g6_stacking"],
    extra_nongraph_models=["random_forest", "mlp", "xgboost", "lightgbm"],
    extra_tuned_models=["xgboost_tuned", "lightgbm_tuned"],       # small search on the validation split only
    extra_gnn_models=["gcn", "graphsage", "residual_gcn"],
    # (graph_type, k, sparsify, retention) used in groups 1-3 ("best graph GNNs" from the earlier runs)
    extra_graph_configs=[("cs2", 3, "none", 1.0), ("fingerprint", 3, "none", 1.0), ("cs2", 3, "black_hole", 0.3)],
    extra_g4_graphs=[("cs2", 3, "none", 1.0)],                    # graph used with each descriptor feature set
    extra_g5_graphs=[("dedup_cs2", 3, "none", 1.0), ("desc", 3, "none", 1.0), ("desc_dedup", 3, "none", 1.0)],
    extra_stack_graphs=["cs2", "dedup_cs2"],
    best_feature_set=None,       # None = chosen automatically from VALIDATION R2 after group 4; or {"with_pore": "...", "without_pore": "..."}
    gbm_early_stopping=50,
)
# Examples of quick overrides (uncomment):
# CFG["seeds"] = list(range(10))
# CFG["k_values"] = [2, 3, 5]
# CFG["retentions"] = [0.2, 0.3, 0.5, 0.7]
# CFG["gnn_models"] = ["gcn", "residual_gcn"]
# CFG["scalings"] = ["none", "standard"]        # standardising features is a cheap, often large win for GNNs
# CFG["language_models"] = ["chemberta", "molt5"]

if PROFILE.startswith("extra"):
    CFG.update(block_non_graph=False, block_no_edge=False, block_graph_full=False, block_sparsify=False, block_extra=True)
else:
    # ---- MolT5 run: only MolT5 (ChemBERTa results already exist), both test sets ----
    CFG["language_models"] = ["molt5"]
    CFG["test_sets"] = ["random15", "cs2_hubs"]

print(f"PROFILE={PROFILE}")
for key, val in CFG.items():
    print(f"  {key}: {val}")

PROFILE=extra
  language_models: []
  test_sets: ['random15', 'cs2_hubs']
  scalings: ['none']
  feature_sets: []
  node_features: []
  non_graph_models: []
  gnn_models: []
  graph_types: []
  k_values: []
  hybrid_betas: [0.5]
  sparsify_graphs: []
  sparsify_k_values: []
  sparsify_methods: []
  retentions: []
  bh_weights: [(0.33, 0.33, 0.33)]
  n_subsample: None
  seeds: [0, 1, 2, 3, 4]
  epochs: 200
  block_non_graph: False
  block_no_edge: False
  block_graph_full: False
  block_sparsify: False
  hidden: 64
  lr: 0.005
  weight_decay: 0.001
  patience: 20
  rf_trees: 100
  knn_neighbors: 5
  mlp_hidden: (128, 64)
  split_state: 42
  test_frac: 0.15
  val_frac: 0.15
  max_hours: None
  block_extra: True
  extra_groups: ['g1_baselines', 'g2_log', 'g3_nopore', 'g4_descriptors', 'g5_graphs', 'g6_stacking']
  extra_nongraph_models: ['random_forest', 'mlp', 'xgboost', 'lightgbm']
  extra_tuned_models: ['xgboost_tuned', 'lightgbm_tuned']
  extra_gnn_models: ['gcn', 'graphsage', 'residu

## 3. Data and features

In [4]:
INVALID_SMILES, INVALID_REPLACEMENT = "F[Si](F)(F)(F)(F)F", "c1ccccc1"


def derive_pld_category(pld):
    if pd.isna(pld) or pld < 2.4:
        return "nonporous"
    if pld < 4.0:
        return "small pore"
    if pld < 8.0:
        return "medium pore"
    return "large pore"


def load_data(path, n_subsample=None):
    df = pd.read_csv(path, index_col=0)
    df["linker SMILES"] = df["linker SMILES"].replace(INVALID_SMILES, INVALID_REPLACEMENT)
    df = df.rename(columns={"linker SMILES": "linker_smiles", "Pore Limiting Diameter": "pld_value"})
    df["refcode"] = df.index
    df["pld_category"] = df["pld_value"].apply(derive_pld_category)
    if n_subsample:
        df = df.sample(n=min(n_subsample, len(df)), random_state=0)
    return df.reset_index(drop=True)


def file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


_MORGAN = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)


def morgan_matrix(smiles_series):
    """1024-bit Morgan fingerprints (radius 2); invalid SMILES -> all-zero row. Computed once per
    distinct SMILES (only ~3,200 distinct linkers exist for 14,296 MOFs)."""
    cache = {}
    for smi in smiles_series.unique():
        mol = Chem.MolFromSmiles(smi)
        cache[smi] = (np.zeros(1024, np.float32) if mol is None or mol.GetNumAtoms() == 0
                      else _MORGAN.GetFingerprintAsNumPy(mol).astype(np.float32))
    return np.stack([cache[s] for s in smiles_series.values]).astype(np.float32)


def build_fingerprint_features(df, fp):
    """1024 Morgan bits + 2 pore-geometry columns + one-hot metal. PLD itself is NOT a feature."""
    pore = df[["Largest Cavity Diameter", "Largest Free Sphere"]].values.astype(np.float32)
    metals = sorted(df["metal"].dropna().unique().tolist())
    index = {m: i for i, m in enumerate(metals)}
    one_hot = np.zeros((len(df), len(metals)), np.float32)
    for row, m in enumerate(df["metal"].values):
        one_hot[row, index[m]] = 1.0
    return np.concatenate([fp, pore, one_hot], axis=1).astype(np.float32)


def embed_with_hf(smiles_list, lm, batch_size=64):
    from transformers import AutoModel, AutoTokenizer, T5EncoderModel
    tok = AutoTokenizer.from_pretrained(HF_IDS[lm])
    model = (T5EncoderModel if lm == "molt5" else AutoModel).from_pretrained(HF_IDS[lm]).to(DEVICE).eval()
    chunks = []
    with torch.no_grad():
        for i in tqdm(range(0, len(smiles_list), batch_size), desc=f"embedding ({lm})"):
            enc = tok(smiles_list[i:i + batch_size], padding=True, truncation=True, return_tensors="pt").to(DEVICE)
            hidden = model(**enc).last_hidden_state
            mask = enc["attention_mask"].unsqueeze(-1).float()
            chunks.append(((hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)).cpu().numpy())
    return np.concatenate(chunks).astype(np.float32)


def get_embeddings(df, lm):
    """Looks every SMILES up in the uploaded .npz (keyed by SMILES text, so row order and
    subsampling do not matter); only SMILES missing from the file are computed with the model."""
    lookup = {}
    path = EMB_PATHS[lm]
    if os.path.exists(path):
        cached = np.load(path, allow_pickle=True)
        for smi, vec in zip(cached["smiles"], cached["embeddings"]):
            lookup.setdefault(str(smi), vec)
        print(f"[{lm}] loaded {len(lookup)} cached embeddings from {path}")
    missing = [s for s in df["linker_smiles"].unique() if s not in lookup]
    if missing:
        print(f"[{lm}] computing {len(missing)} missing embeddings with {HF_IDS[lm]} (needs internet)")
        for smi, vec in zip(missing, embed_with_hf(missing, lm)):
            lookup[smi] = vec
    return np.stack([lookup[s] for s in df["linker_smiles"].values]).astype(np.float32)


df = load_data(DATA_PATH, CFG["n_subsample"])
N = len(df)
LABELS = df["pld_value"].values.astype(np.float32)
DATA_SHA = file_sha256(DATA_PATH)
FP = morgan_matrix(df["linker_smiles"])
X_FP = build_fingerprint_features(df, FP)
EMB = {lm: get_embeddings(df, lm) for lm in CFG["language_models"]}
print(f"MOFs={N}  fingerprint-features={X_FP.shape}  data sha256={DATA_SHA[:16]}…  PLD std={LABELS.std():.3f}")

MOFs=14296  fingerprint-features=(14296, 1079)  data sha256=e23a42e742099b4c…  PLD std=2.931


## 4. Graph construction  (cs2 = Case Study 2 graph · fingerprint · semantic · hybrid)

* `cs2`: 0.9 × Tanimoto(linker) + 0.1 × same-metal indicator, k nearest neighbours (Case Study 2 / MOFGalaxyNet style)
* `fingerprint`: Tanimoto(linker fingerprints) only
* `semantic`: cosine similarity of the language-model embeddings
* `hybrid`: β × cosine(embeddings) + (1 − β) × Tanimoto

In [9]:
BLOCK = 1500


def _normalize(a):
    norms = np.linalg.norm(a, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return a / norms


def _tanimoto_block(rows):
    inter = FP[rows] @ FP.T
    union = FP[rows].sum(1)[:, None] + FP.sum(1)[None, :] - inter
    union[union == 0] = 1.0
    return inter / union


def knn_graph(sim_fn, k):
    """Undirected top-k graph from a blocked similarity function (never builds the N x N matrix)."""
    g = nx.Graph()
    g.add_nodes_from(range(N))
    kk = min(k, N - 1)
    for start in range(0, N, BLOCK):
        rows = slice(start, min(start + BLOCK, N))
        sim = sim_fn(rows)
        local = np.arange(sim.shape[0])
        sim[local, local + start] = -1.0
        top = np.argpartition(-sim, kk - 1, axis=1)[:, :kk]
        for r in range(sim.shape[0]):
            for c in top[r]:
                if sim[r, c] > -1.0:
                    g.add_edge(start + r, int(c), weight=float(sim[r, c]))
    return g


_METAL_CODES = pd.factorize(df["metal"])[0]
_GRAPH_CACHE = {}


def get_graph(graph_type, k, beta=0.5, lm="-"):
    if graph_type == "none":
        g = nx.Graph()
        g.add_nodes_from(range(N))
        return g
    key = (graph_type, k, beta if graph_type == "hybrid" else 0.0, lm if graph_type in ("semantic", "hybrid") else "-")
    if key not in _GRAPH_CACHE:
        t = time.time()
        if graph_type == "cs2":
            fn = lambda r: 0.9 * _tanimoto_block(r) + 0.1 * (_METAL_CODES[r][:, None] == _METAL_CODES[None, :]).astype(np.float32)
        elif graph_type == "fingerprint":
            fn = _tanimoto_block
        elif graph_type == "semantic":
            en = _normalize(EMB[lm])
            fn = lambda r: en[r] @ en.T
        elif graph_type == "hybrid":
            en = _normalize(EMB[lm])
            fn = lambda r: beta * (en[r] @ en.T) + (1 - beta) * _tanimoto_block(r)
        else:
            raise ValueError(graph_type)
        _GRAPH_CACHE[key] = knn_graph(fn, k)
        print(f"built graph {key}: {_GRAPH_CACHE[key].number_of_edges()} edges in {time.time() - t:.0f}s")
    return _GRAPH_CACHE[key]

## 5. Test sets and splits

* `random15` — 15 % of the MOFs chosen at random with a fixed seed (the Step 5 test set)
* `cs2_hubs` — Case Study 2's test set: ~4 % of the highest-"gravity" MOFs in each Louvain community of the `cs2` k=3 graph

In [10]:
def gravity_per_community(graph, communities, weights):
    wd, wb, ws = weights
    gravity, fixed_test = {}, set()
    for community in communities:
        sub = graph.subgraph(community)
        if sub.number_of_nodes() == 0:
            continue
        deg = nx.degree_centrality(sub)
        bet = nx.betweenness_centrality(sub, normalized=True)
        wsum = {n: sum(d["weight"] for _, _, d in sub.edges(n, data=True)) for n in sub.nodes()}
        nodes = list(community)
        norm = lambda vals: MinMaxScaler().fit_transform(np.array(vals).reshape(-1, 1)).flatten()
        nd, nb, nw = norm([deg[n] for n in nodes]), norm([bet[n] for n in nodes]), norm([wsum[n] for n in nodes])
        for i, n in enumerate(nodes):
            gravity[n] = wd * nd[i] + wb * nb[i] + ws * nw[i]
        degrees = {n: sub.degree(n) for n in community}
        eligible = [n for n, d in degrees.items() if d > 2] or ([max(degrees, key=degrees.get)] if degrees else [])
        if eligible:
            ranked = sorted(eligible, key=lambda n: gravity[n], reverse=True)
            fixed_test.update(ranked[:max(1, int(0.04 * len(community)))])
    return gravity, fixed_test


_TEST_CACHE = {}


def get_test_nodes(name):
    if name not in _TEST_CACHE:
        if name == "random15":
            rng = np.random.default_rng(CFG["split_state"])
            _TEST_CACHE[name] = set(rng.choice(N, size=max(1, round(CFG["test_frac"] * N)), replace=False).tolist())
        elif name == "cs2_hubs":
            g = get_graph("cs2", 3)
            comms = louvain_communities(g, seed=42)
            _TEST_CACHE[name] = gravity_per_community(g, comms, (0.33, 0.33, 0.33))[1]
        else:
            raise ValueError(name)
        print(f"test set '{name}': {len(_TEST_CACHE[name])} MOFs")
    return _TEST_CACHE[name]


def make_splits(test_nodes, seed):
    rng = np.random.RandomState(seed)
    remaining = np.array([i for i in range(N) if i not in test_nodes])
    rng.shuffle(remaining)
    n_val = max(1, int(len(remaining) * CFG["val_frac"]))
    return remaining[n_val:], remaining[:n_val], np.array(sorted(test_nodes))

## 6. Sparsification: Black Hole, random, k-center

In [11]:
def black_hole_nodes(graph, gravity, communities, threshold, fixed_test, pld_categories):
    g = graph.copy()
    selected = set(fixed_test)
    n_total = g.number_of_nodes()
    target = int(max(0.2 * n_total, (1 - threshold) * n_total))
    target_non_test = max(0, target - len(fixed_test))
    sizes = [len(c) for c in communities]
    total = sum(sizes) or 1
    targets = [max(1, int(target_non_test * (s / total))) for s in sizes]
    diff = target_non_test - sum(targets)
    for i in range(abs(diff)):
        j = i % len(targets)
        targets[j] = max(0, targets[j] + (1 if diff > 0 else -1))
    remove = []
    for community, tcount in zip(communities, targets):
        nodes = [n for n in community if n in gravity and n not in fixed_test]
        if not nodes:
            continue
        props = pld_categories.iloc[nodes].value_counts(normalize=True).to_dict()
        for cat, prop in props.items():
            cand = [n for n in nodes if pld_categories.iloc[n] == cat]
            selected.update(sorted(cand, key=lambda n: gravity[n], reverse=True)[:max(1, int(tcount * prop))])
        remove.extend(n for n in nodes if n not in selected)
    g.remove_nodes_from(remove)
    return g


def prune_edges(graph, edge_threshold, fixed_test):
    edges = [(u, v, d["weight"]) for u, v, d in graph.edges(data=True)]
    if not edges:
        return graph
    test_edges = [e for e in edges if e[0] in fixed_test or e[1] in fixed_test]
    other = [e for e in edges if e[0] not in fixed_test and e[1] not in fixed_test]
    best = {}
    for u, v, w in test_edges:
        for n in (u, v):
            if n in fixed_test and (n not in best or w > best[n][2]):
                best[n] = (u, v, w)
    keep_test = list({(min(u, v), max(u, v)): (u, v, w) for u, v, w in best.values()}.values())
    n_keep = max(0, max(10, int((1 - edge_threshold) * len(edges))) - len(keep_test))
    other.sort(key=lambda e: e[2], reverse=True)
    out = nx.Graph()
    out.add_nodes_from(graph.nodes())
    out.add_weighted_edges_from(keep_test + other[:n_keep])
    return out


def restore_test_nodes(pruned, original, fixed_test):
    """Add back any test node pruning dropped (with its original edges) so every method is
    evaluated on the identical test set."""
    missing = fixed_test - set(pruned.nodes())
    if not missing:
        return pruned
    out = pruned.copy()
    out.add_nodes_from(missing)
    for node in missing:
        for nb, d in original.adj[node].items():
            if nb in out:
                out.add_edge(node, nb, **d)
    return out


def kcenter_nodes(features, n_keep, seed, required):
    rng = np.random.default_rng(seed)
    selected = list(dict.fromkeys(required)) or [int(rng.integers(0, N))]
    min_dist = np.full(N, np.inf)
    for c in selected:
        min_dist = np.minimum(min_dist, np.linalg.norm(features - features[c], axis=1))
    while len(selected) < n_keep and len(selected) < N:
        nxt = int(np.argmax(min_dist))
        if min_dist[nxt] == 0:
            break
        selected.append(nxt)
        min_dist = np.minimum(min_dist, np.linalg.norm(features - features[nxt], axis=1))
        min_dist[nxt] = 0.0
    short = min(n_keep, N) - len(selected)
    if short > 0:  # many MOFs share one linker -> fewer distinct points than the budget: fill randomly
        rest = np.setdiff1d(np.arange(N), np.array(selected, dtype=int))
        selected.extend(int(i) for i in rng.choice(rest, size=short, replace=False))
    return selected[:n_keep]


def kcenter_space(graph_type, lm):
    fpn = _normalize(FP)
    if graph_type in ("cs2", "fingerprint"):
        return fpn
    if graph_type == "semantic":
        return _normalize(EMB[lm])
    return np.concatenate([fpn, _normalize(EMB[lm])], axis=1)


_BH_CACHE = {}
_KC_CACHE = {}


def sparsify(graph, graph_type, k, beta, lm, method, retention, seed, test_name, bh_weights):
    test_nodes = get_test_nodes(test_name)
    if method == "none":
        return graph
    if method == "black_hole":
        key = (graph_type, k, beta, lm, retention, bh_weights, test_name)
        if key not in _BH_CACHE:  # deterministic (Louvain seed 42), so computed once, reused for every seed
            communities = louvain_communities(graph, seed=42)
            gravity, own_test = gravity_per_community(graph, communities, bh_weights)
            pruned = black_hole_nodes(graph, gravity, communities, 1 - retention, own_test, df["pld_category"])
            pruned = prune_edges(pruned, 1 - retention, own_test)
            _BH_CACHE[key] = restore_test_nodes(pruned, graph, test_nodes)
        return _BH_CACHE[key]
    n_keep = max(len(test_nodes), round(retention * N))
    if method == "random":
        rng = np.random.default_rng(seed)
        cand = [n for n in graph.nodes() if n not in test_nodes]
        extra = rng.choice(cand, size=min(max(0, n_keep - len(test_nodes)), len(cand)), replace=False).tolist()
        return graph.subgraph(set(test_nodes) | set(extra)).copy()
    if method == "k_center":
        ck = (graph_type, lm, n_keep, seed, test_name)  # selection does not depend on k or on the model: compute once, reuse
        if ck not in _KC_CACHE:
            _KC_CACHE[ck] = kcenter_nodes(kcenter_space(graph_type, lm), n_keep, seed, sorted(test_nodes))
        chosen = _KC_CACHE[ck]
        return graph.subgraph(set(chosen)).copy()
    raise ValueError(method)

## 7. Models

`residual_gcn` is the new variant: it starts as a plain MLP and the graph term is multiplied by a learnable
gate that starts at 0, so each MOF keeps trusting its own features unless neighbours demonstrably help.

In [12]:
def tgt_fwd(y, target):
    return np.log1p(np.clip(y, 0, None)).astype(np.float32) if target == "log1p" else y


def tgt_inv(z, target):
    return np.clip(np.expm1(z), 0, None).astype(np.float32) if target == "log1p" else z


class GCN(nn.Module):
    def __init__(self, d_in, d_h, d_out):
        super().__init__()
        self.conv1, self.conv2 = GCNConv(d_in, d_h), GCNConv(d_h, d_out)

    def forward(self, x, ei, ew=None):
        h = F.dropout(F.relu(self.conv1(x, ei, ew)), p=0.3, training=self.training)
        return self.conv2(h, ei, ew)


class GraphSAGE(nn.Module):
    """Mean aggregation over neighbours + self loop (same hand-rolled layer as the repo)."""

    def __init__(self, d_in, d_h, d_out):
        super().__init__()
        self.l1, self.l2 = nn.Linear(d_in, d_h), nn.Linear(d_h, d_out)

    def _norm_adj(self, n, ei, ew, device):
        loops = torch.stack([torch.arange(n, device=device)] * 2)
        if ew is None:
            ew = torch.ones(ei.size(1), device=device)
            self_w = torch.ones(n, device=device)
        else:
            self_w = torch.ones(n, device=device) * ew.mean()
        ei2, ew2 = torch.cat([ei, loops], 1), torch.cat([ew, self_w])
        adj = torch.sparse_coo_tensor(ei2, ew2, (n, n))
        deg = torch.sparse.sum(adj, dim=1).to_dense().clamp(min=1.0)
        return torch.sparse_coo_tensor(ei2, ew2 / deg[ei2[0]], (n, n))

    def forward(self, x, ei, ew=None):
        A = self._norm_adj(x.size(0), ei, ew, x.device)
        h = F.dropout(F.relu(self.l1(torch.sparse.mm(A, x))), p=0.3, training=self.training)
        return self.l2(torch.sparse.mm(A, h))


class GAT(nn.Module):
    def __init__(self, d_in, d_h, d_out, heads=12):
        super().__init__()
        self.conv1 = GATConv(d_in, d_h, heads=heads, dropout=0.3)
        self.norm = nn.LayerNorm(d_h * heads)
        self.conv2 = GATConv(d_h * heads, d_out, heads=1, concat=False, dropout=0.3)

    def forward(self, x, ei, ew=None):
        h = F.dropout(F.relu(self.norm(self.conv1(x, ei))), p=0.3, training=self.training)
        return self.conv2(h, ei)


class ResidualGCN(nn.Module):
    def __init__(self, d_in, d_h, d_out):
        super().__init__()
        self.lin_in, self.conv = nn.Linear(d_in, d_h), GCNConv(d_h, d_h)
        self.gate = nn.Parameter(torch.zeros(1))  # 0 -> starts as a pure MLP
        self.lin_out = nn.Linear(d_h, d_out)

    def forward(self, x, ei, ew=None):
        h = F.dropout(F.relu(self.lin_in(x)), p=0.3, training=self.training)
        h = h + self.gate * F.relu(self.conv(h, ei, ew))
        return self.lin_out(h)


GNN_CLASSES = {"gcn": GCN, "graphsage": GraphSAGE, "gat": GAT, "residual_gcn": ResidualGCN}


def edge_arrays(graph):
    edges = list(graph.edges(data=True))
    if not edges:
        return np.zeros((2, 0), np.int64), np.zeros(0, np.float32)
    src = [u for u, v, d in edges] + [v for u, v, d in edges]
    dst = [v for u, v, d in edges] + [u for u, v, d in edges]
    return np.array([src, dst]), np.array([d["weight"] for u, v, d in edges] * 2, np.float32)


def train_eval_gnn(model_name, feats, graph, test_nodes, seed, target="raw", extended=False):
    train_idx, val_idx, test_idx = make_splits(test_nodes, seed)
    ei_np, ew_np = edge_arrays(graph)
    x = torch.tensor(feats, dtype=torch.float, device=DEVICE)
    y = torch.tensor(tgt_fwd(LABELS, target), dtype=torch.float, device=DEVICE)
    ei = torch.tensor(ei_np, dtype=torch.long, device=DEVICE)
    ew = torch.tensor(ew_np, dtype=torch.float, device=DEVICE) if ew_np.size else None
    tr, va, te = (torch.tensor(a, dtype=torch.long, device=DEVICE) for a in (train_idx, val_idx, test_idx))

    model = GNN_CLASSES[model_name](feats.shape[1], CFG["hidden"], 1).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
    best_val, best_state, wait, epochs_run = float("inf"), None, 0, 0
    t0 = time.time()
    for epoch in range(CFG["epochs"]):
        model.train()
        opt.zero_grad()
        loss = F.mse_loss(model(x, ei, ew).squeeze(-1)[tr], y[tr])
        loss.backward()
        opt.step()
        model.eval()
        with torch.no_grad():
            val_loss = F.mse_loss(model(x, ei, ew).squeeze(-1)[va], y[va]).item()
        epochs_run = epoch + 1
        if val_loss < best_val:
            best_val, wait = val_loss, 0
            best_state = {k_: v.clone() for k_, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= CFG["patience"]:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        full = model(x, ei, ew).squeeze(-1).cpu().numpy()
    pred = tgt_inv(full[test_idx], target)
    true = LABELS[test_idx]
    out = metrics_ext(true, pred) if extended else metrics(true, pred)
    if extended:
        out["val_r2"] = float(r2_score(LABELS[val_idx], tgt_inv(full[val_idx], target)))
    out.update(epochs_run=epochs_run, best_val_mse=best_val, train_seconds=time.time() - t0)
    if model_name == "residual_gcn":
        out["learned_gate"] = float(model.gate.item())
    return out


def metrics_ext(true, pred):
    """Metrics on the original scale plus the same three metrics on the log1p scale."""
    out = metrics(true, pred)
    lt, lp = np.log1p(np.clip(true, 0, None)), np.log1p(np.clip(pred, 0, None))
    out.update(mae_log=float(mean_absolute_error(lt, lp)), rmse_log=float(np.sqrt(mean_squared_error(lt, lp))),
               r2_log=float(r2_score(lt, lp)))
    return out


def metrics(true, pred):
    return {"mae": float(mean_absolute_error(true, pred)),
            "rmse": float(np.sqrt(mean_squared_error(true, pred))),
            "r2": float(r2_score(true, pred))}


def train_eval_classic(model_name, feats, test_nodes, seed):
    train_idx, _val_idx, test_idx = make_splits(test_nodes, seed)
    t0 = time.time()
    if model_name == "random_forest":
        model = RandomForestRegressor(n_estimators=CFG["rf_trees"], random_state=seed, n_jobs=-1)
    elif model_name == "mlp":
        model = MLPRegressor(hidden_layer_sizes=CFG["mlp_hidden"], random_state=seed,
                             early_stopping=True, n_iter_no_change=20, max_iter=500)
    elif model_name == "knn":
        model = KNeighborsRegressor(n_neighbors=CFG["knn_neighbors"], n_jobs=-1)
    else:
        raise ValueError(model_name)
    model.fit(feats[train_idx], LABELS[train_idx])
    out = metrics(LABELS[test_idx], model.predict(feats[test_idx]))
    out["train_seconds"] = time.time() - t0
    return out

## 7b. Extra experiments (only run by PROFILE `extra_smoke` / `extra`)

Groups (each on both test sets, seeds 0-4; the target is raw PLD unless it says log1p):

| group | what it tests |
|---|---|
| `g1_baselines` | XGBoost / LightGBM (fixed settings and a small validation-only search) next to Random Forest and MLP, plus the GNNs with and without edges on the plain fingerprint features |
| `g2_log` | the same models trained on log1p(PLD); metrics are reported on the original Å scale and on the log scale |
| `g3_nopore` | everything again **without the LCD and LFS columns** (does the graph add anything once the easy geometric signal is gone?) |
| `g4_descriptors` | metal descriptors (added to, or instead of, the one-hot) and RDKit linker descriptors, alone and together, with and without LCD/LFS |
| `g5_graphs` | on the best feature sets (chosen by **validation** R², never test): linker-level de-duplicated graph, descriptor-space graph, and both GCN variants including the residual GCN; each is compared with the no-edge GNN on the same features |
| `g6_stacking` | XGBoost with label-free graph-smoothed features added, against XGBoost alone |

Nothing uses PLD as an input; hyper-parameters and early stopping use only the validation split.


In [13]:
# ============================ EXTRA EXPERIMENTS (only used by PROFILE "extra" / "extra_smoke") ============================
# Everything below is additive: the old blocks and their run ids are untouched. Nothing here ever uses PLD (or a
# function of it) as an input; hyper-parameters and early stopping use the validation split only.

PER_RUN_PATH = os.path.join(OUT_DIR, "per_run_results.csv")   # same path the run cell uses (needed here by pick_best_feature_sets)

# ---------- features: RDKit linker descriptors and metal descriptors ----------
LINK_ID, LINK_SMILES = pd.factorize(df["linker_smiles"])          # one id per distinct linker (~3,200)
LINK_FIRST = np.array([np.flatnonzero(LINK_ID == u)[0] for u in range(len(LINK_SMILES))])
LINK_MOFS = [np.flatnonzero(LINK_ID == u) for u in range(len(LINK_SMILES))]
LINK_DESC_NAMES = ["mol_weight", "heavy_atoms", "ring_count", "aromatic_rings", "rotatable_bonds", "tpsa", "logp",
                   "frac_csp3", "radius_of_gyration", "asphericity"]
METAL_DESC_NAMES = ["atomic_number", "atomic_mass", "atomic_radius", "electronegativity_pauling", "ionic_radius",
                    "valence_electrons"]
DESC_INFO = {}


def _conformer_shape(mol, seed):
    """Radius of gyration and asphericity from ONE ETKDG conformer (fixed seed). (nan, nan) if embedding fails."""
    try:
        mh = Chem.AddHs(mol)
        params = AllChem.ETKDGv3()
        params.randomSeed = int(seed)
        params.useRandomCoords = True
        if AllChem.EmbedMolecule(mh, params) != 0:
            return np.nan, np.nan
        return float(rdMolDescriptors.CalcRadiusOfGyration(mh)), float(rdMolDescriptors.CalcAsphericity(mh))
    except Exception:
        return np.nan, np.nan


def compute_linker_descriptors(smiles_list, seed):
    rows, n_bad_smiles, n_no_conformer = [], 0, 0
    for smi in tqdm(smiles_list, desc="linker descriptors"):
        mol = Chem.MolFromSmiles(smi)
        if mol is None or mol.GetNumAtoms() == 0:
            rows.append([np.nan] * len(LINK_DESC_NAMES))
            n_bad_smiles += 1
            continue
        base = [Descriptors.MolWt(mol), mol.GetNumHeavyAtoms(), rdMolDescriptors.CalcNumRings(mol),
                rdMolDescriptors.CalcNumAromaticRings(mol), rdMolDescriptors.CalcNumRotatableBonds(mol),
                rdMolDescriptors.CalcTPSA(mol), Descriptors.MolLogP(mol), rdMolDescriptors.CalcFractionCSP3(mol)]
        rg, asph = _conformer_shape(mol, seed) if mol.GetNumHeavyAtoms() <= 150 else (np.nan, np.nan)
        n_no_conformer += int(np.isnan(rg))
        rows.append(base + [rg, asph])
    arr = np.array(rows, dtype=float)
    medians = np.nanmedian(arr, axis=0)                      # logged fallback: median over distinct linkers
    arr = np.where(np.isnan(arr), medians[None, :], arr)
    info = dict(n_distinct_linkers=len(smiles_list), n_unparseable_smiles=n_bad_smiles,
                n_no_conformer_rg_asphericity_imputed=n_no_conformer, fallback="column median over distinct linkers",
                conformer_seed=int(seed))
    return arr.astype(np.float32), info


# Pauling electronegativity, used only if neither pymatgen nor mendeleev is available (RDKit has no such table).
_PAULING_FALLBACK = {"Ag": 1.93, "Al": 1.61, "Ba": 0.89, "Be": 1.57, "Bi": 2.02, "Ca": 1.00, "Cd": 1.69, "Ce": 1.12,
                     "Co": 1.88, "Cr": 1.66, "Cs": 0.79, "Cu": 1.90, "Dy": 1.22, "Er": 1.24, "Eu": 1.2, "Fe": 1.83,
                     "Ga": 1.81, "Gd": 1.2, "Hf": 1.3, "Hg": 2.00, "Ho": 1.23, "In": 1.78, "K": 0.82, "La": 1.10,
                     "Li": 0.98, "Lu": 1.27, "Mg": 1.31, "Mn": 1.55, "Mo": 2.16, "Na": 0.93, "Nd": 1.14, "Ni": 1.91,
                     "Pb": 2.33, "Pr": 1.13, "Rb": 0.82, "Ru": 2.2, "Sb": 2.05, "Sc": 1.36, "Sm": 1.17, "Sn": 1.96,
                     "Sr": 0.95, "Tb": 1.2, "Th": 1.3, "Ti": 1.54, "Tl": 1.62, "Tm": 1.25, "U": 1.38, "V": 1.63,
                     "W": 2.36, "Y": 1.22, "Yb": 1.1, "Zn": 1.65, "Zr": 1.33}


def _f(x):
    try:
        return float(x)
    except Exception:
        return np.nan


def metal_descriptor_table(symbols):
    """symbols -> (array [n_symbols x 6], source). Order = METAL_DESC_NAMES. Tiers: pymatgen, mendeleev, RDKit table
    (no ionic radius there: that column becomes the median = constant)."""
    try:
        from pymatgen.core import Element
        rows = []
        for s in symbols:
            e = Element(s)
            ox = [o for o in (e.common_oxidation_states or ()) if o in (e.ionic_radii or {})]
            ionic = (e.ionic_radii or {}).get(ox[0] if ox else (min(e.ionic_radii) if e.ionic_radii else None))
            group = e.group
            rows.append([e.Z, _f(e.atomic_mass), _f(e.atomic_radius), _f(e.X), _f(ionic),
                         group if group <= 12 else group - 10])
        return np.array(rows, dtype=float), "pymatgen"
    except Exception as exc_pm:
        print("pymatgen unavailable (", type(exc_pm).__name__, ") -> trying mendeleev")
    try:
        from mendeleev import element
        rows = []
        for s in symbols:
            e = element(s)
            rows.append([e.atomic_number, _f(e.atomic_weight), _f(e.atomic_radius), _f(e.en_pauling),
                         _f(e.ionic_radii[0].ionic_radius / 100) if e.ionic_radii else np.nan, _f(e.nvalence())])
        return np.array(rows, dtype=float), "mendeleev"
    except Exception as exc_md:
        print("mendeleev unavailable (", type(exc_md).__name__, ") -> RDKit periodic table + hard-coded Pauling values")
    pt = Chem.GetPeriodicTable()
    rows = []
    for s in symbols:
        z = pt.GetAtomicNumber(s)
        rows.append([z, pt.GetAtomicWeight(z), pt.GetRcovalent(z), _PAULING_FALLBACK.get(s, np.nan), np.nan,
                     pt.GetNOuterElecs(z)])
    return np.array(rows, dtype=float), "rdkit+hardcoded_pauling (no ionic radius)"


def build_extra_descriptors():
    global LDESC_UNIQUE, LDESC_MOF, MDESC
    LDESC_UNIQUE, info = compute_linker_descriptors(list(LINK_SMILES), 0xF00D)
    LDESC_MOF = LDESC_UNIQUE[LINK_ID]
    symbols = sorted(df["metal"].dropna().unique().tolist())
    table, source = metal_descriptor_table(symbols)
    medians = np.nanmedian(table, axis=0)
    table = np.where(np.isnan(table), np.where(np.isnan(medians), 0.0, medians)[None, :], table)
    lookup = {s: table[i] for i, s in enumerate(symbols)}
    MDESC = np.stack([lookup[m] for m in df["metal"].values]).astype(np.float32)
    info.update(metal_descriptor_source=source, n_metals=len(symbols), metal_descriptor_columns=METAL_DESC_NAMES,
                linker_descriptor_columns=LINK_DESC_NAMES)
    DESC_INFO.update(info)
    print("descriptor info:", json.dumps(info, default=str))


if CFG.get("block_extra"):
    build_extra_descriptors()

_SCALED = {}


def _scaled_dense(name, test_name):
    """Descriptor columns standardised with statistics from NON-test MOFs only (no labels involved)."""
    key = (name, test_name)
    if key not in _SCALED:
        raw = MDESC if name == "mdesc" else LDESC_MOF
        non_test = np.array([i for i in range(N) if i not in get_test_nodes(test_name)])
        _SCALED[key] = StandardScaler().fit(raw[non_test]).transform(raw).astype(np.float32)
    return _SCALED[key]


N_BITS = FP.shape[1]
FS_BASE, FS_NOPORE = "fp+pore+m1h", "fp+m1h"   # FS_BASE is exactly the established "fingerprint" feature matrix


def extra_feature_matrix(name, test_name):
    """name = components joined by '+': fp (Morgan bits), pore (LCD, LFS), m1h (metal one-hot),
    mdesc (metal descriptors), ldesc (RDKit linker descriptors)."""
    parts = []
    for c in name.split("+"):
        if c == "fp":
            parts.append(X_FP[:, :N_BITS])
        elif c == "pore":
            parts.append(X_FP[:, N_BITS:N_BITS + 2])
        elif c == "m1h":
            parts.append(X_FP[:, N_BITS + 2:])
        elif c in ("mdesc", "ldesc"):
            parts.append(_scaled_dense(c, test_name))
        else:
            raise ValueError(f"unknown feature component {c!r} in {name!r}")
    return np.ascontiguousarray(np.hstack(parts), dtype=np.float32)


def descriptor_feature_sets():
    out = []
    for pore in (True, False):
        p = ["fp"] + (["pore"] if pore else [])
        out += ["+".join(p + ["m1h", "mdesc"]), "+".join(p + ["mdesc"]),
                "+".join(p + ["m1h", "ldesc"]), "+".join(p + ["m1h", "mdesc", "ldesc"])]
    return out


# ---------- extra graphs: de-duplicated at the distinct-linker level, and descriptor-space graphs ----------
EXTRA_GRAPHS = ("dedup_cs2", "desc", "desc_dedup")
_XG_CACHE = {}


def _knn_from_matrix(sim, k):
    sim = sim.copy()
    np.fill_diagonal(sim, -np.inf)
    kk = min(k, sim.shape[0] - 1)
    return np.argpartition(-sim, kk - 1, axis=1)[:, :kk]


def _zscore(a):
    sd = a.std(0)
    sd[sd == 0] = 1.0
    return (a - a.mean(0)) / sd


def _dedup_graph(link_sim, k, same_metal_weight):
    """Link every MOF to ONE MOF of each of the k most similar OTHER linkers (same metal if it exists, else the
    lowest index; the pore columns are never used, so the no-pore ablation stays clean). MOFs that share a linker
    are therefore never each other's neighbours."""
    nbrs = _knn_from_matrix(link_sim, k)
    first_with_metal = {}
    for i in range(N):
        first_with_metal.setdefault((LINK_ID[i], _METAL_CODES[i]), i)
    g = nx.Graph()
    g.add_nodes_from(range(N))
    for i in range(N):
        u = LINK_ID[i]
        for v in nbrs[u]:
            j = first_with_metal.get((v, _METAL_CODES[i]), int(LINK_MOFS[v][0]))
            w = (1 - same_metal_weight) * link_sim[u, v] + same_metal_weight * float(_METAL_CODES[i] == _METAL_CODES[j])
            if not g.has_edge(i, j):
                g.add_edge(i, j, weight=float(w))
    return g


def get_graph_extra(graph_type, k):
    key = (graph_type, k)
    if key in _XG_CACHE:
        return _XG_CACHE[key]
    t = time.time()
    if graph_type == "dedup_cs2":      # same 0.9 Tanimoto + 0.1 same-metal idea as the CS2 graph, but linker-level
        fpu = FP[LINK_FIRST]
        inter = fpu @ fpu.T
        union = fpu.sum(1)[:, None] + fpu.sum(1)[None, :] - inter
        union[union == 0] = 1.0
        g = _dedup_graph(inter / union, k, 0.1)
    elif graph_type == "desc_dedup":   # nearest linkers in the standardised RDKit-descriptor space
        z = _zscore(LDESC_UNIQUE)
        sq = (z ** 2).sum(1)
        d = np.sqrt(np.maximum(sq[:, None] + sq[None, :] - 2 * z @ z.T, 0))
        g = _dedup_graph(1.0 / (1.0 + d), k, 0.0)
    elif graph_type == "desc":         # nearest MOFs in standardised [linker descriptors + metal descriptors]
        z = np.hstack([_zscore(LDESC_MOF), _zscore(MDESC)]).astype(np.float32)
        sq = (z ** 2).sum(1)
        fn = lambda r: 1.0 / (1.0 + np.sqrt(np.maximum(sq[r][:, None] + sq[None, :] - 2 * z[r] @ z.T, 0)))
        g = knn_graph(fn, k)
    else:
        raise ValueError(graph_type)
    _XG_CACHE[key] = g
    print(f"built extra graph {key}: {g.number_of_edges()} edges in {time.time() - t:.0f}s")
    return g


def graph_for(spec):
    gt = spec["graph_type"]
    if gt == "none":
        return get_graph("none", 0)
    if gt in EXTRA_GRAPHS:
        return get_graph_extra(gt, spec["k"])
    base = get_graph(gt, spec["k"], spec["beta"], spec["language_model"])
    wt = tuple(float(v) for v in spec["bh_weights"].split("/")) if spec["bh_weights"] != "-" else None
    return sparsify(base, gt, spec["k"], spec["beta"], spec["language_model"], spec["sparsify"], spec["retention"],
                    spec["seed"], spec["test_set"], wt)


def smooth_features(feats, graph):
    """Label-free graph-smoothed features: weighted mean of each node's neighbours' non-fingerprint columns
    (pore / metal / descriptors; the fingerprint bits if there is nothing else)."""
    cols = feats[:, N_BITS:] if feats.shape[1] > N_BITS else feats[:, :N_BITS]
    (src, dst), w = edge_arrays(graph)
    sums, deg = np.zeros_like(cols), np.zeros(N, np.float32)
    if len(src):
        np.add.at(sums, dst, cols[src] * w[:, None])
        np.add.at(deg, dst, w)
    smooth = np.where(deg[:, None] > 0, sums / np.maximum(deg, 1e-9)[:, None], cols)
    return np.hstack([feats, smooth.astype(np.float32)])


# ---------- gradient-boosted baselines (XGBoost, LightGBM), fixed settings + small validation-only search ----------
GBM_DEFAULT = {
    "xgboost": dict(n_estimators=2000, learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.5,
                    min_child_weight=1, reg_lambda=1.0),
    "lightgbm": dict(n_estimators=2000, learning_rate=0.05, num_leaves=31, subsample=0.8, subsample_freq=1,
                     colsample_bytree=0.5, min_child_samples=20, reg_lambda=0.0),
}
GBM_GRID = {   # 6 candidates each, scored on the VALIDATION split of seed 0 only
    "xgboost": [dict(max_depth=d, colsample_bytree=c) for d in (4, 6, 8) for c in (0.3, 0.6)],
    "lightgbm": [dict(num_leaves=n, colsample_bytree=c) for n in (15, 31, 63) for c in (0.3, 0.6)],
}
_HP_CACHE = {}


def make_gbm(base, hp, seed):
    if base == "xgboost":
        return xgb.XGBRegressor(**hp, tree_method="hist", early_stopping_rounds=CFG["gbm_early_stopping"],
                                random_state=seed, n_jobs=-1, verbosity=0)
    return lgb.LGBMRegressor(**hp, random_state=seed, n_jobs=-1, verbose=-1)


def fit_gbm(model, base, X_tr, y_tr, X_va, y_va):
    if base == "xgboost":
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
        return int(model.best_iteration) + 1
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[lgb.early_stopping(CFG["gbm_early_stopping"], verbose=False)])
    return int(model.best_iteration_)


def tuned_hp(base, feats, feat_key, target, test_name):
    key = (base, feat_key, target, test_name)
    if key not in _HP_CACHE:
        tr, va, _ = make_splits(get_test_nodes(test_name), 0)     # validation only; the test set is never touched
        y_tr, y_va = tgt_fwd(LABELS[tr], target), tgt_fwd(LABELS[va], target)
        best = None
        for cand in GBM_GRID[base]:
            hp = dict(GBM_DEFAULT[base], **cand)
            m = make_gbm(base, hp, 0)
            fit_gbm(m, base, feats[tr], y_tr, feats[va], y_va)
            rmse = float(np.sqrt(mean_squared_error(y_va, m.predict(feats[va]))))
            if best is None or rmse < best[0]:
                best = (rmse, hp)
        _HP_CACHE[key] = best[1]
        print(f"  tuned {base} on {feat_key}/{target}/{test_name}: {best[1]} (val rmse {best[0]:.3f})")
    return _HP_CACHE[key]


def train_eval_tabular(model_name, feats, test_nodes, seed, target, feat_key, test_name):
    train_idx, val_idx, test_idx = make_splits(test_nodes, seed)
    y_tr, y_va = tgt_fwd(LABELS[train_idx], target), tgt_fwd(LABELS[val_idx], target)
    t0, extra = time.time(), {}
    base = model_name.replace("_tuned", "").replace("_smooth", "")
    if base in ("xgboost", "lightgbm"):
        hp = tuned_hp(base, feats, feat_key, target, test_name) if model_name.endswith("_tuned") else GBM_DEFAULT[base]
        model = make_gbm(base, hp, seed)
        extra["best_iteration"] = fit_gbm(model, base, feats[train_idx], y_tr, feats[val_idx], y_va)
        extra["hparams"] = json.dumps(hp)
    elif model_name == "random_forest":
        model = RandomForestRegressor(n_estimators=CFG["rf_trees"], random_state=seed, n_jobs=-1).fit(feats[train_idx], y_tr)
    elif model_name == "mlp":
        model = MLPRegressor(hidden_layer_sizes=CFG["mlp_hidden"], random_state=seed, early_stopping=True,
                             n_iter_no_change=20, max_iter=500).fit(feats[train_idx], y_tr)
    elif model_name == "knn":
        model = KNeighborsRegressor(n_neighbors=CFG["knn_neighbors"], n_jobs=-1).fit(feats[train_idx], y_tr)
    else:
        raise ValueError(model_name)
    pred = tgt_inv(model.predict(feats[test_idx]), target)
    out = metrics_ext(LABELS[test_idx], pred)
    out["val_r2"] = float(r2_score(LABELS[val_idx], tgt_inv(model.predict(feats[val_idx]), target)))
    out.update(extra, train_seconds=time.time() - t0)
    return out


def execute_extra(spec):
    set_all_seeds(spec["seed"])
    ts, target = spec["test_set"], spec["target"]
    test_nodes = get_test_nodes(ts)
    feats = extra_feature_matrix(spec["node_features"], ts)
    graph = None
    if spec["block"] == "extra_tab":
        if spec["model"].endswith("_smooth"):          # stacking: graph-smoothed (label-free) features added for XGBoost
            graph = graph_for(spec)
            feats = smooth_features(feats, graph)
        res = train_eval_tabular(spec["model"], feats, test_nodes, spec["seed"], target, spec["node_features"], ts)
    else:
        graph = graph_for(spec)
        res = train_eval_gnn(spec["model"], feats, graph, test_nodes, spec["seed"], target=target, extended=True)
    row = dict(spec)
    row.update(res)
    row.update(n_test=len(test_nodes), n_features=feats.shape[1], device=str(DEVICE),
               graph_nodes_with_edges=int(sum(1 for _, d in graph.degree() if d > 0)) if graph is not None else 0,
               graph_nodes=graph.number_of_nodes() if graph is not None else N,
               graph_edges=graph.number_of_edges() if graph is not None else 0,
               data_sha256=DATA_SHA[:16], finished_at=datetime.now().isoformat(timespec="seconds"))
    return row


# ---------- which experiments to run ----------
def pick_best_feature_sets():
    """Best feature set WITH and WITHOUT the pore columns, chosen by mean VALIDATION R2 of default XGBoost (raw target)
    over all seeds and test sets of groups 1, 3 and 4 -- never by test score. None until those runs are all finished."""
    forced = CFG.get("best_feature_set")
    if forced:
        return forced
    if not os.path.exists(PER_RUN_PATH):
        return None
    pr = pd.read_csv(PER_RUN_PATH)
    if "val_r2" not in pr.columns:
        return None
    pr = pr[(pr["block"] == "extra_tab") & (pr["model"] == "xgboost") & (pr["target"] == "raw") & pr["val_r2"].notna()]
    cands = [FS_BASE, FS_NOPORE] + descriptor_feature_sets()
    need = len(CFG["seeds"]) * len(CFG["test_sets"])
    counts = pr.groupby("node_features").size()
    if any(counts.get(c, 0) < need for c in cands):
        return None
    mean_val = pr.groupby("node_features")["val_r2"].mean()
    with_pore = max((c for c in cands if "pore" in c), key=lambda c: mean_val[c])
    without_pore = max((c for c in cands if "pore" not in c), key=lambda c: mean_val[c])
    return {"with_pore": with_pore, "without_pore": without_pore}


def make_extra_specs(cfg):
    specs, seen = [], set()

    def add(group, block, model, feats, target, ts, graph_type="none", k=0, sparsify="none", retention=1.0):
        for seed in cfg["seeds"]:
            spec = dict(block=block, model=model, node_features=feats, scaling="none", graph_type=graph_type, k=k,
                        beta=0.0, language_model="-", sparsify=sparsify, retention=retention,
                        bh_weights="0.33/0.33/0.33" if sparsify == "black_hole" else "-", test_set=ts, seed=seed,
                        target=target)
            rid = hashlib.md5(json.dumps(spec, sort_keys=True).encode()).hexdigest()[:16]
            if rid not in seen:
                seen.add(rid)
                specs.append(dict(spec, run_id=rid, group=group))

    groups, gm = cfg["extra_groups"], cfg["extra_gnn_models"]
    ng, tuned = cfg["extra_nongraph_models"], cfg["extra_tuned_models"]

    def tab(group, feats, target, ts, models):
        for m in models:
            add(group, "extra_tab", m, feats, target, ts)

    def gnn(group, feats, target, ts, graphs):
        for m in gm:                                   # no edges = the reference every graph is compared with
            add(group, "extra_gnn", m, feats, target, ts)
        for gt, k, sp, ret in graphs:
            for m in gm:
                add(group, "extra_gnn", m, feats, target, ts, gt, k, sp, ret)

    best = pick_best_feature_sets() if ("g5_graphs" in groups or "g6_stacking" in groups) else None
    for ts in cfg["test_sets"]:
        if "g1_baselines" in groups:
            tab("g1_baselines", FS_BASE, "raw", ts, ng + tuned)
            gnn("g1_baselines", FS_BASE, "raw", ts, cfg["extra_graph_configs"])
        if "g2_log" in groups:
            for fs in (FS_BASE, FS_NOPORE):
                tab("g2_log", fs, "log1p", ts, ng)
                gnn("g2_log", fs, "log1p", ts, cfg["extra_graph_configs"])
        if "g3_nopore" in groups:
            tab("g3_nopore", FS_NOPORE, "raw", ts, ng + tuned)
            gnn("g3_nopore", FS_NOPORE, "raw", ts, cfg["extra_graph_configs"])
        if "g4_descriptors" in groups:
            for fs in descriptor_feature_sets():
                tab("g4_descriptors", fs, "raw", ts, ng)
                gnn("g4_descriptors", fs, "raw", ts, cfg["extra_g4_graphs"])
        if best:
            for fs in dict.fromkeys(best.values()):
                if "g5_graphs" in groups:
                    gnn("g5_graphs", fs, "raw", ts, cfg["extra_g5_graphs"])
                if "g6_stacking" in groups:
                    for gt in cfg["extra_stack_graphs"]:
                        add("g6_stacking", "extra_tab", "xgboost_smooth", fs, "raw", ts, gt, 3)
    return specs


linker descriptors:   0%|          | 0/3193 [00:00<?, ?it/s]

descriptor info: {"n_distinct_linkers": 3193, "n_unparseable_smiles": 1, "n_no_conformer_rg_asphericity_imputed": 7, "fallback": "column median over distinct linkers", "conformer_seed": 61453, "metal_descriptor_source": "pymatgen", "n_metals": 53, "metal_descriptor_columns": ["atomic_number", "atomic_mass", "atomic_radius", "electronegativity_pauling", "ionic_radius", "valence_electrons"], "linker_descriptor_columns": ["mol_weight", "heavy_atoms", "ring_count", "aromatic_rings", "rotatable_bonds", "tpsa", "logp", "frac_csp3", "radius_of_gyration", "asphericity"]}


## 8. Build the list of runs

In [14]:
def feature_matrix(name, lm, scaling, test_nodes):
    if name == "fingerprint":
        feats = X_FP
    elif name == "embedding":
        feats = EMB[lm]
    elif name == "fingerprint+embedding":
        feats = np.hstack([X_FP, EMB[lm]])
    else:
        raise ValueError(name)
    if scaling == "standard":  # statistics from non-test MOFs only; no labels involved
        non_test = np.array([i for i in range(N) if i not in test_nodes])
        feats = StandardScaler().fit(feats[non_test]).transform(feats).astype(np.float32)
    return np.ascontiguousarray(feats, dtype=np.float32)


def uses_lm(graph_type, node_features):
    return graph_type in ("semantic", "hybrid") or "embedding" in node_features


def make_specs(cfg):
    specs, seen = [], set()

    def add(**kw):
        base = dict(block=None, model=None, node_features="fingerprint", scaling="none", graph_type="none", k=0,
                    beta=0.0, language_model="-", sparsify="none", retention=1.0, bh_weights="-", test_set=None)
        base.update(kw)
        if base["graph_type"] != "hybrid":
            base["beta"] = 0.0
        for seed in cfg["seeds"]:
            spec = dict(base, seed=seed)
            rid = hashlib.md5(json.dumps(spec, sort_keys=True).encode()).hexdigest()[:16]
            if rid not in seen:
                seen.add(rid)
                specs.append(dict(spec, run_id=rid))

    def lms(graph_type, node_features):
        return cfg["language_models"] if uses_lm(graph_type, node_features) else ["-"]

    def betas(graph_type):
        return cfg["hybrid_betas"] if graph_type == "hybrid" else [0.0]

    for ts in cfg["test_sets"]:
        for sc in cfg["scalings"]:
            if cfg["block_non_graph"]:
                for m in cfg["non_graph_models"]:
                    for fs in cfg["feature_sets"]:
                        for lm in lms("none", fs):
                            add(block="non_graph", model=m, node_features=fs, scaling=sc, language_model=lm, test_set=ts)
            if cfg["block_no_edge"]:
                for m in cfg["gnn_models"]:
                    for fs in cfg["feature_sets"]:
                        for lm in lms("none", fs):
                            add(block="no_edge", model=m, node_features=fs, scaling=sc, language_model=lm, test_set=ts)
            if cfg["block_graph_full"]:
                for gt in cfg["graph_types"]:
                    for k in cfg["k_values"]:
                        for beta in betas(gt):
                            for nf in cfg["node_features"]:
                                for lm in lms(gt, nf):
                                    for m in cfg["gnn_models"]:
                                        add(block="graph_full", model=m, node_features=nf, scaling=sc, graph_type=gt, k=k,
                                            beta=beta, language_model=lm, test_set=ts)
            if cfg["block_sparsify"]:
                for gt in cfg["sparsify_graphs"]:
                    for k in cfg["sparsify_k_values"]:
                        for beta in betas(gt):
                            for nf in cfg["node_features"]:
                                for lm in lms(gt, nf):
                                    for method in cfg["sparsify_methods"]:
                                        for ret in cfg["retentions"]:
                                            weightings = cfg["bh_weights"] if method == "black_hole" else [None]
                                            for w in weightings:
                                                wlabel = "/".join(str(v) for v in w) if w else "-"
                                                for m in cfg["gnn_models"]:
                                                    add(block="sparsify", model=m, node_features=nf, scaling=sc, graph_type=gt,
                                                        k=k, beta=beta, language_model=lm, sparsify=method, retention=ret,
                                                        bh_weights=wlabel, test_set=ts)
    if cfg.get("block_extra"):
        specs += make_extra_specs(cfg)
    return specs


SPECS = make_specs(CFG)
spec_df = pd.DataFrame(SPECS)
print(f"TOTAL RUNS: {len(SPECS)}")
print(spec_df.groupby(["block", "model"]).size().unstack(fill_value=0))

TOTAL RUNS: 1480
model      gcn  graphsage  lightgbm  lightgbm_tuned  mlp  random_forest  \
block                                                                     
extra_gnn  320        320         0               0    0              0   
extra_tab    0          0       120              20  120            120   

model      residual_gcn  xgboost  xgboost_tuned  
block                                            
extra_gnn           320        0              0  
extra_tab             0      120             20  


## 9. Run (resumable)

Each finished run is appended to `per_run_results.csv` immediately. Re-running this cell skips runs that are already there.

In [15]:
os.makedirs(OUT_DIR, exist_ok=True)
PER_RUN_PATH = os.path.join(OUT_DIR, "per_run_results.csv")
done_ids = set(pd.read_csv(PER_RUN_PATH)["run_id"]) if os.path.exists(PER_RUN_PATH) else set()
todo = [s for s in SPECS if s["run_id"] not in done_ids]
print(f"already done: {len(done_ids & {s['run_id'] for s in SPECS})}  |  to do now: {len(todo)}")


# fixed column order, so rows from different model types (RF has no epochs, residual GCN has a gate, ...) line up
PER_RUN_COLUMNS = ["run_id", "block", "model", "node_features", "scaling", "graph_type", "k", "beta", "language_model",
                   "sparsify", "retention", "bh_weights", "test_set", "seed", "mae", "rmse", "r2", "train_seconds",
                   "epochs_run", "best_val_mse", "learned_gate", "n_test", "n_features", "graph_nodes",
                   "graph_nodes_with_edges", "graph_edges", "device", "data_sha256", "finished_at",
                   "group", "target", "val_r2", "mae_log", "rmse_log", "r2_log", "hparams", "best_iteration"]

if os.path.exists(PER_RUN_PATH):   # an older results file has fewer columns: widen it once so rows keep lining up
    _old = pd.read_csv(PER_RUN_PATH)
    if list(_old.columns) != PER_RUN_COLUMNS:
        _old.reindex(columns=PER_RUN_COLUMNS).to_csv(PER_RUN_PATH, index=False)


def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def execute(spec):
    if spec["block"].startswith("extra"):
        return execute_extra(spec)
    set_all_seeds(spec["seed"])
    test_nodes = get_test_nodes(spec["test_set"])
    lm = spec["language_model"]
    feats = feature_matrix(spec["node_features"], lm, spec["scaling"], test_nodes)
    if spec["block"] == "non_graph":
        res = train_eval_classic(spec["model"], feats, test_nodes, spec["seed"])
        graph = None
    else:
        base = get_graph(spec["graph_type"], spec["k"], spec["beta"], lm)
        wt = tuple(float(v) for v in spec["bh_weights"].split("/")) if spec["bh_weights"] != "-" else None
        graph = sparsify(base, spec["graph_type"], spec["k"], spec["beta"], lm, spec["sparsify"], spec["retention"],
                         spec["seed"], spec["test_set"], wt)
        res = train_eval_gnn(spec["model"], feats, graph, test_nodes, spec["seed"])
    row = dict(spec)
    row.update(res)
    row.update(n_test=len(test_nodes), n_features=feats.shape[1], device=str(DEVICE),
               graph_nodes_with_edges=int(sum(1 for _, d in graph.degree() if d > 0)) if graph is not None else 0,
               graph_nodes=graph.number_of_nodes() if graph is not None else N,
               graph_edges=graph.number_of_edges() if graph is not None else 0,
               data_sha256=DATA_SHA[:16], finished_at=datetime.now().isoformat(timespec="seconds"))
    return row


START = time.time()
ATTEMPTED = set()


def run_specs(todo):
    pbar = tqdm(todo, desc="runs")
    for spec in pbar:
        if CFG["max_hours"] and (time.time() - START) / 3600 > CFG["max_hours"]:
            print(f"max_hours={CFG['max_hours']} reached — stopping. Re-run this cell to resume.")
            return True
        ATTEMPTED.add(spec["run_id"])
        try:
            row = execute(spec)
        except Exception as exc:  # keep going; the failure is recorded and can be retried by re-running
            print(f"FAILED {spec['model']} {spec['graph_type']} k={spec['k']} {spec['sparsify']}: {type(exc).__name__}: {exc}")
            with open(os.path.join(OUT_DIR, "failed_runs.log"), "a") as fh:
                fh.write(f"{datetime.now().isoformat()} {json.dumps(spec)} {type(exc).__name__}: {exc}\n")
            continue
        pd.DataFrame([row]).reindex(columns=PER_RUN_COLUMNS).to_csv(
            PER_RUN_PATH, mode="a", header=not os.path.exists(PER_RUN_PATH), index=False)
        pbar.set_postfix(model=row["model"], graph=row["graph_type"], r2=round(row["r2"], 3))
    return False


stopped = run_specs(todo)
# The extra profiles choose their later groups (graphs / stacking on the best descriptor set) from the VALIDATION
# results of the earlier groups, so the run list is rebuilt after each pass; older profiles simply find nothing new.
for _round in range(3):
    if stopped or not CFG.get("block_extra"):
        break
    SPECS = make_specs(CFG)
    done_ids = set(pd.read_csv(PER_RUN_PATH)["run_id"]) if os.path.exists(PER_RUN_PATH) else set()
    todo = [s for s in SPECS if s["run_id"] not in done_ids and s["run_id"] not in ATTEMPTED]
    if not todo:
        break
    print(f"next pass: {len(todo)} more runs; best feature sets chosen on validation R2: {pick_best_feature_sets()}")
    stopped = run_specs(todo)
print(f"finished in {(time.time() - START) / 60:.1f} min")


already done: 0  |  to do now: 1480


runs:   0%|          | 0/1480 [00:00<?, ?it/s]

test set 'random15': 2144 MOFs
  tuned xgboost on fp+pore+m1h/raw/random15: {'n_estimators': 2000, 'learning_rate': 0.05, 'max_depth': 8, 'subsample': 0.8, 'colsample_bytree': 0.3, 'min_child_weight': 1, 'reg_lambda': 1.0} (val rmse 0.894)
  tuned lightgbm on fp+pore+m1h/raw/random15: {'n_estimators': 2000, 'learning_rate': 0.05, 'num_leaves': 63, 'subsample': 0.8, 'subsample_freq': 1, 'colsample_bytree': 0.6, 'min_child_samples': 20, 'reg_lambda': 0.0} (val rmse 0.961)
built graph ('cs2', 3, 0.0, '-'): 32988 edges in 3s
built graph ('fingerprint', 3, 0.0, '-'): 36576 edges in 3s
  tuned xgboost on fp+m1h/raw/random15: {'n_estimators': 2000, 'learning_rate': 0.05, 'max_depth': 8, 'subsample': 0.8, 'colsample_bytree': 0.3, 'min_child_weight': 1, 'reg_lambda': 1.0} (val rmse 1.784)
  tuned lightgbm on fp+m1h/raw/random15: {'n_estimators': 2000, 'learning_rate': 0.05, 'num_leaves': 63, 'subsample': 0.8, 'subsample_freq': 1, 'colsample_bytree': 0.6, 'min_child_samples': 20, 'reg_lambda': 0

runs:   0%|          | 0/220 [00:00<?, ?it/s]

built extra graph ('dedup_cs2', 3): 39816 edges in 0s
built extra graph ('desc', 3): 31682 edges in 3s
built extra graph ('desc_dedup', 3): 39593 edges in 0s
finished in 48.5 min


## 10. Aggregate, compare with the no-edge reference, export

In [16]:
per_run = pd.read_csv(PER_RUN_PATH)
for _col, _default in (("target", "raw"), ("group", "-")):   # rows from older runs predate these columns
    per_run[_col] = per_run[_col].fillna(_default) if _col in per_run else _default
GROUP = ["block", "model", "node_features", "scaling", "graph_type", "k", "beta", "language_model",
         "sparsify", "retention", "bh_weights", "test_set", "target"]
agg = (per_run.groupby(GROUP, dropna=False)
       .agg(n_seeds=("seed", "nunique"), mae_mean=("mae", "mean"), mae_std=("mae", "std"),
            rmse_mean=("rmse", "mean"), rmse_std=("rmse", "std"), r2_mean=("r2", "mean"), r2_std=("r2", "std"),
            seconds_mean=("train_seconds", "mean"), graph_nodes=("graph_nodes", "mean"),
            graph_edges=("graph_edges", "mean"))
       .reset_index())

# "Does the graph help or hurt?": R2 of each graph run minus R2 of the SAME model/features/scaling/test set with no edges.
# The graph's own language model is ignored in the match unless the node features contain embeddings.
agg["ref_lm"] = np.where(agg["node_features"].str.contains("embedding"), agg["language_model"], "-")
ref_keys = ["model", "node_features", "scaling", "ref_lm", "test_set", "target"]
is_no_edge = (agg["block"] == "no_edge") | ((agg["block"] == "extra_gnn") & (agg["graph_type"] == "none"))
is_graph = agg["block"].isin(["graph_full", "sparsify"]) | ((agg["block"] == "extra_gnn") & (agg["graph_type"] != "none"))
no_edge = agg[is_no_edge][ref_keys + ["r2_mean", "mae_mean"]].rename(
    columns={"r2_mean": "r2_no_edge", "mae_mean": "mae_no_edge"})
vs = agg[is_graph].merge(no_edge, on=ref_keys, how="left")
vs["r2_change_vs_no_edge"] = vs["r2_mean"] - vs["r2_no_edge"]
vs["verdict"] = np.where(vs["r2_change_vs_no_edge"] > 0.01, "graph helps",
                         np.where(vs["r2_change_vs_no_edge"] < -0.01, "graph hurts", "about the same"))

best_nongraph = agg[agg["block"].isin(["non_graph", "extra_tab"])].sort_values("r2_mean", ascending=False).head(1)
print("Best non-graph model:")
print(best_nongraph[["model", "node_features", "r2_mean", "mae_mean"]].to_string(index=False))
print("\nTop 15 GNN configurations by R2:")
print(agg[~agg["block"].isin(["non_graph", "extra_tab"])].sort_values("r2_mean", ascending=False).head(15)[
    ["block", "model", "graph_type", "k", "sparsify", "retention", "node_features", "r2_mean", "r2_std", "mae_mean"]
].to_string(index=False))
print("\nGraph vs no-edge verdict counts:")
print(vs["verdict"].value_counts().to_string())

agg.to_csv(os.path.join(OUT_DIR, "aggregated_results.csv"), index=False)
vs.to_csv(os.path.join(OUT_DIR, "graph_vs_no_graph.csv"), index=False)

Best non-graph model:
        model           node_features  r2_mean  mae_mean
random_forest fp+pore+m1h+mdesc+ldesc 0.904973  0.512654

Top 15 GNN configurations by R2:
    block        model graph_type  k sparsify  retention           node_features  r2_mean   r2_std  mae_mean
extra_gnn    graphsage       none  0     none        1.0 fp+pore+m1h+mdesc+ldesc 0.859450 0.001113  0.689468
extra_gnn residual_gcn       none  0     none        1.0 fp+pore+m1h+mdesc+ldesc 0.857758 0.006446  0.692541
extra_gnn residual_gcn        cs2  3     none        1.0       fp+pore+m1h+ldesc 0.857384 0.003956  0.696214
extra_gnn residual_gcn       desc  3     none        1.0       fp+pore+m1h+ldesc 0.856987 0.004613  0.695308
extra_gnn residual_gcn        cs2  3     none        1.0 fp+pore+m1h+mdesc+ldesc 0.856518 0.004806  0.697044
extra_gnn residual_gcn       none  0     none        1.0       fp+pore+m1h+ldesc 0.856224 0.005125  0.694527
extra_gnn    graphsage       none  0     none        1.0 fp+pore+m1

## 10b. Extra experiments: comparison table, paired differences, Q1-Q3

In [17]:
ex = per_run[per_run["block"].astype(str).str.startswith("extra")].copy()
extra_table = extra_paired = pd.DataFrame()
if len(ex):
    def _desc_label(fs):
        m, l = ("mdesc" in fs), ("ldesc" in fs)
        metal = "metal descriptors + one-hot" if (m and "m1h" in fs) else ("metal descriptors (replace one-hot)" if m else "metal one-hot")
        return metal + (" + RDKit linker descriptors" if l else "")

    ex["pore"] = np.where(ex["node_features"].str.contains("pore"), "with LCD/LFS", "without LCD/LFS")
    ex["descriptors"] = ex["node_features"].map(_desc_label)
    ex["graph"] = np.where(ex["graph_type"] == "none", "no edges",
                           ex["graph_type"] + " k=" + ex["k"].astype(str) + " " + ex["sparsify"].replace("none", "full")
                           + np.where(ex["sparsify"] != "none", " " + ex["retention"].astype(str), ""))
    ex["model_graph"] = np.where(ex["block"] == "extra_gnn", ex["model"] + " | " + ex["graph"], ex["model"])
    KEYS = ["test_set", "target", "pore", "descriptors", "node_features", "model_graph"]
    extra_table = (ex.groupby(KEYS)
                   .agg(n_seeds=("seed", "nunique"), r2_mean=("r2", "mean"), r2_std=("r2", "std"),
                        mae_mean=("mae", "mean"), mae_std=("mae", "std"), rmse_mean=("rmse", "mean"),
                        rmse_std=("rmse", "std"), r2_log_mean=("r2_log", "mean"), mae_log_mean=("mae_log", "mean"),
                        val_r2_mean=("val_r2", "mean"))
                   .reset_index().sort_values(["test_set", "pore", "r2_mean"], ascending=[True, True, False]))
    extra_table.to_csv(os.path.join(OUT_DIR, "extra_comparison_table.csv"), index=False)

    def sel(**kw):
        d = ex
        for key, val in kw.items():
            d = d[d[key].isin(val) if isinstance(val, (list, tuple, set)) else d[key] == val]
        return d

    def paired(a, b, label, ts):
        """Per-seed R2 difference a - b on identical (test set, seed) splits."""
        m = a[["seed", "r2"]].merge(b[["seed", "r2"]], on="seed", suffixes=("_a", "_b"))
        if m.empty:
            return None
        diff = m["r2_a"] - m["r2_b"]
        noise = max(a["r2"].std(), b["r2"].std())
        return dict(comparison=label, test_set=ts, n_seeds=len(m), r2_a=a["r2"].mean(), r2_b=b["r2"].mean(),
                    mean_diff=diff.mean(), std_diff=diff.std(), seed_spread=noise,
                    verdict=("within seed noise" if abs(diff.mean()) <= noise else ("A better" if diff.mean() > 0 else "B better")))

    rows = []
    for ts in sorted(ex["test_set"].unique()):
        T = lambda **kw: sel(test_set=ts, **kw)
        # Q1: gradient boosting vs Random Forest (raw target)
        for fs in (FS_BASE, FS_NOPORE):
            rf = T(block="extra_tab", model="random_forest", node_features=fs, target="raw")
            for m in ("xgboost", "lightgbm", "xgboost_tuned", "lightgbm_tuned"):
                r = paired(T(block="extra_tab", model=m, node_features=fs, target="raw"), rf, f"Q1 {m} - random_forest [{fs}]", ts)
                rows.append(r)
        # log target vs raw target (scored on the original Angstrom scale)
        for fs in (FS_BASE, FS_NOPORE):
            for m in ["random_forest", "mlp", "xgboost", "lightgbm"]:
                rows.append(paired(T(block="extra_tab", model=m, node_features=fs, target="log1p"),
                                   T(block="extra_tab", model=m, node_features=fs, target="raw"), f"LOG-TARGET {m} - raw [{fs}]", ts))
            for m in sorted(ex[ex.block == "extra_gnn"]["model"].unique()):
                rows.append(paired(T(block="extra_gnn", model=m, node_features=fs, target="log1p", graph_type="none"),
                                   T(block="extra_gnn", model=m, node_features=fs, target="raw", graph_type="none"), f"LOG-TARGET no-edge {m} - raw [{fs}]", ts))
        # Q3: new descriptors vs the plain feature set (same pore status)
        for base_fs, pre in ((FS_BASE, "fp+pore+"), (FS_NOPORE, "fp+")):
            for fs in sorted(set(ex.node_features) - {FS_BASE, FS_NOPORE}):
                if not fs.startswith(pre) or ("pore" in fs) != ("pore" in base_fs):
                    continue
                for m in ["random_forest", "xgboost", "lightgbm", "mlp"]:
                    rows.append(paired(T(block="extra_tab", model=m, node_features=fs, target="raw"),
                                       T(block="extra_tab", model=m, node_features=base_fs, target="raw"), f"Q3 {m}: {fs} - {base_fs}", ts))
                for m in sorted(ex[ex.block == "extra_gnn"]["model"].unique()):
                    rows.append(paired(T(block="extra_gnn", model=m, node_features=fs, target="raw", graph_type="none"),
                                       T(block="extra_gnn", model=m, node_features=base_fs, target="raw", graph_type="none"), f"Q3 no-edge {m}: {fs} - {base_fs}", ts))
        # graph vs the SAME model with no edges, same features
        g = T(block="extra_gnn", target="raw")
        for (fs, m, gtype, k, sp, ret), grp in g[g.graph_type != "none"].groupby(["node_features", "model", "graph_type", "k", "sparsify", "retention"]):
            rows.append(paired(grp, T(block="extra_gnn", model=m, node_features=fs, target="raw", graph_type="none"),
                               f"GRAPH {m} {gtype} k={k} {sp} {ret} - no edges [{fs}]", ts))
        # stacking
        for fs in sorted(ex[ex.model == "xgboost_smooth"]["node_features"].unique()):
            for gt in sorted(ex[ex.model == "xgboost_smooth"]["graph_type"].unique()):
                rows.append(paired(T(model="xgboost_smooth", node_features=fs, graph_type=gt), T(block="extra_tab", model="xgboost", node_features=fs, target="raw"),
                                   f"STACK xgboost+smoothed({gt}) - xgboost [{fs}]", ts))
    extra_paired = pd.DataFrame([r for r in rows if r])
    extra_paired.to_csv(os.path.join(OUT_DIR, "extra_paired_differences.csv"), index=False)

    # Q2: once LCD/LFS are removed, how close does the best graph GNN get to the best non-graph model?
    q2 = []
    for ts in sorted(ex["test_set"].unique()):
        for pore in ("without LCD/LFS", "with LCD/LFS"):
            d = extra_table[(extra_table.test_set == ts) & (extra_table.pore == pore) & (extra_table.target == "raw")]
            ng = d[~d.model_graph.str.contains(r"\|") & ~d.model_graph.str.contains("smooth")].sort_values("r2_mean", ascending=False)
            gr = d[d.model_graph.str.contains(r"\| (?!no edges)")].sort_values("r2_mean", ascending=False)
            ne = d[d.model_graph.str.contains("no edges")].sort_values("r2_mean", ascending=False)
            if len(ng) and len(gr):
                q2.append(dict(test_set=ts, pore=pore, best_non_graph=f"{ng.iloc[0].model_graph} [{ng.iloc[0].node_features}]",
                               best_non_graph_r2=ng.iloc[0].r2_mean, best_no_edge_gnn_r2=ne.iloc[0].r2_mean if len(ne) else np.nan,
                               best_graph_gnn=f"{gr.iloc[0].model_graph} [{gr.iloc[0].node_features}]", best_graph_gnn_r2=gr.iloc[0].r2_mean,
                               gap=ng.iloc[0].r2_mean - gr.iloc[0].r2_mean, within_0_02=(ng.iloc[0].r2_mean - gr.iloc[0].r2_mean) <= 0.02))
    q2 = pd.DataFrame(q2)
    q2.to_csv(os.path.join(OUT_DIR, "extra_q2_graph_vs_nongraph.csv"), index=False)

    pd.set_option("display.width", 250, "display.max_colwidth", 70)
    print("=== Q1/log/Q3/graph/stack paired differences (R2, a - b; 'within seed noise' = |mean diff| <= seed std) ===")
    print(extra_paired.round(4).to_string(index=False))
    print("\n=== Q2: best graph GNN vs best non-graph model (selection by mean test R2 = a reporting choice) ===")
    print(q2.round(4).to_string(index=False))
    print("\n=== Top 15 per test set (raw target) ===")
    for ts in sorted(ex["test_set"].unique()):
        print(ts)
        print(extra_table[(extra_table.test_set == ts) & (extra_table.target == "raw")].head(15)[
            ["pore", "descriptors", "model_graph", "r2_mean", "r2_std", "mae_mean", "rmse_mean"]].round(4).to_string(index=False))
else:
    print("No extra-experiment rows yet (PROFILE must be 'extra' or 'extra_smoke').")


=== Q1/log/Q3/graph/stack paired differences (R2, a - b; 'within seed noise' = |mean diff| <= seed std) ===
                                                                comparison test_set  n_seeds    r2_a   r2_b  mean_diff  std_diff  seed_spread           verdict
                                  Q1 xgboost - random_forest [fp+pore+m1h] cs2_hubs        5  0.8769 0.8876    -0.0107    0.0076       0.0069          B better
                                 Q1 lightgbm - random_forest [fp+pore+m1h] cs2_hubs        5  0.8684 0.8876    -0.0192    0.0132       0.0129          B better
                            Q1 xgboost_tuned - random_forest [fp+pore+m1h] cs2_hubs        5  0.8822 0.8876    -0.0054    0.0042       0.0044          B better
                           Q1 lightgbm_tuned - random_forest [fp+pore+m1h] cs2_hubs        5  0.8681 0.8876    -0.0195    0.0082       0.0087          B better
                                       Q1 xgboost - random_forest [fp+m1h] cs2_hubs        5

## 11. Metadata, README, Excel and zip  — download `mofsemnet_results.zip`

In [18]:
def _pkg(name):
    try:
        return __import__(name).__version__
    except Exception:
        return "?"


meta = dict(
    created=datetime.now().isoformat(timespec="seconds"), profile=PROFILE,
    config={k_: (list(v) if isinstance(v, tuple) else v) for k_, v in CFG.items()},
    data_file=DATA_PATH, data_sha256=DATA_SHA, n_mofs=N, seeds=CFG["seeds"], device=str(DEVICE),
    gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    python=sys.version.split()[0], platform=platform.platform(),
    packages={p: _pkg(p) for p in ["torch", "torch_geometric", "networkx", "sklearn", "pandas", "numpy", "rdkit", "xgboost", "lightgbm", "pymatgen"]},
    embedding_files={lm: (EMB_PATHS[lm], file_sha256(EMB_PATHS[lm])[:16]) for lm in CFG["language_models"]
                     if os.path.exists(EMB_PATHS[lm])},
    runs_total=len(SPECS), runs_in_csv=len(per_run),
    extra=dict(descriptor_info=DESC_INFO, best_feature_sets=pick_best_feature_sets() if CFG.get('block_extra') else None,
               gbm_default=GBM_DEFAULT, gbm_search_grid=GBM_GRID, tuned_hparams={'|'.join(map(str, k_)): v for k_, v in _HP_CACHE.items()}),
)
with open(os.path.join(OUT_DIR, "run_metadata.json"), "w") as fh:
    json.dump(meta, fh, indent=2, default=str)

readme = f"""MOFSemNet server run — how these files were made and how to regenerate them
=============================================================================
Notebook      : the single notebook this folder came from (profile = {PROFILE})
Data          : {DATA_PATH}  (SHA-256 {DATA_SHA})
Seeds         : {CFG['seeds']}  (every run seeds python, numpy and torch; splits use the same seed)
Test sets     : {CFG['test_sets']}  (random15 = 15% random MOFs, fixed with split_state={CFG['split_state']};
                cs2_hubs = Case Study 2's high-gravity Louvain-community test set)
Target        : continuous Pore Limiting Diameter (Angstrom); metrics MAE, RMSE, R2 on the held-out test set.

Files
  per_run_results.csv     one row per (configuration, seed): all settings, metrics, timing, graph size
  aggregated_results.csv  mean and std over seeds per configuration
  graph_vs_no_graph.csv   each graph result compared with the same model run WITHOUT edges
  run_metadata.json       full configuration, package versions, GPU, data hash
  mofsemnet_results.xlsx  all of the above as sheets

To regenerate: put MOFCSD.csv (and the embeddings .npz files) next to the notebook, set PROFILE = "{PROFILE}"
and the CFG overrides recorded in run_metadata.json, then Run All. Delete per_run_results.csv first
for a clean run (otherwise finished runs are skipped = resume).
"""
with open(os.path.join(OUT_DIR, "README.txt"), "w") as fh:
    fh.write(readme)

xlsx_path = os.path.join(OUT_DIR, "mofsemnet_results.xlsx")
try:
    with pd.ExcelWriter(xlsx_path) as xw:
        agg.to_excel(xw, sheet_name="aggregated", index=False)
        vs.to_excel(xw, sheet_name="graph_vs_no_graph", index=False)
        per_run.to_excel(xw, sheet_name="per_run", index=False)
        for _name, _tab in (("extra_table", extra_table), ("extra_paired", extra_paired)):
            if len(_tab):
                _tab.to_excel(xw, sheet_name=_name, index=False)
        pd.DataFrame([{"key": k_, "value": json.dumps(v, default=str)} for k_, v in meta.items()]).to_excel(
            xw, sheet_name="metadata", index=False)
    print("wrote", xlsx_path)
except Exception as exc:
    print(f"Excel export skipped ({type(exc).__name__}: {exc}). Run `pip install openpyxl` and re-run this cell; the CSVs are complete.")

zip_path = "mofsemnet_results.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for name in sorted(os.listdir(OUT_DIR)):
        zf.write(os.path.join(OUT_DIR, name), arcname=name)
print("download:", os.path.abspath(zip_path))

wrote server_extra_2026-11-20/mofsemnet_results.xlsx
download: /home/jovyan/mofsemnet_results.zip
